# 1. Objetivo

Preparar uma camada Gold simples e reproduzível para o dashboard Lighthouse da LH Nautical. Nesta primeira etapa, o notebook apenas localiza o projeto, carrega integralmente as fontes necessárias e executa validações iniciais, sem filtrar, agregar, corrigir ou exportar dados.

# 2. Arquitetura

`CSVs brutos → notebook pandas → camada Gold em CSV → Power BI → dashboard`

O Power BI consumirá somente os arquivos Gold. As fontes brutas permanecem imutáveis.

# 3. Regras de negócio

- Métricas comerciais usarão `placed_at`, pedidos com `status = paid` e faturamento de `orders.total`.
- O total do pedido não será replicado na granularidade de itens.
- Clientes fiéis serão determinados por pedidos pagos, diversidade mínima de 13 categorias e Top 10 por ticket médio, com desempate por `customer_id`.
- A operação considerará pedidos pagos do canal POS e calendário contínuo, incluindo dias sem venda.
- A previsão de `Bússola de Bordo 702` usará `paid + confirmed`, treino até dezembro de 2025, teste de janeiro a março de 2026 e média móvel recursiva de três meses sem arredondamento. Os dois IDs com esse nome serão preservados.
- A recomendação para `Motor de Popa 1949` usará pedidos pagos, matriz binária cliente-produto, similaridade de cosseno e Top 5 sem o produto de referência. `asdf` será preservado e sinalizado.

# 4. Imports e caminhos

In [29]:
from pathlib import Path

import pandas as pd

current_directory = Path.cwd().resolve()
project_candidates = [current_directory, current_directory.parent]
PROJECT_ROOT = next(
    (candidate for candidate in project_candidates if (candidate / "data" / "1-lh_nautical_csv").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Raiz do projeto não encontrada. Execute o notebook na raiz de lh-nautical ou dentro de notebooks/."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "1-lh_nautical_csv"
GOLD_DATA_DIR = PROJECT_ROOT / "data" / "2-lh_nautical_gold"

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretório de entrada: {RAW_DATA_DIR}")
print(f"Diretório Gold planejado: {GOLD_DATA_DIR}")

Raiz do projeto: /Users/leonardoramos/Documents/dev/Desafios/indicium/desafio-lighthouse-lh-nautical
Diretório de entrada: /Users/leonardoramos/Documents/dev/Desafios/indicium/desafio-lighthouse-lh-nautical/data/1-lh_nautical_csv
Diretório Gold planejado: /Users/leonardoramos/Documents/dev/Desafios/indicium/desafio-lighthouse-lh-nautical/data/2-lh_nautical_gold


# 5. Carregamento das fontes

Os onze CSVs necessários são carregados integralmente em DataFrames nomeados. Nenhuma linha ou coluna é filtrada e nenhum valor é alterado.


In [30]:
orders_df = pd.read_csv(RAW_DATA_DIR / "orders.csv")
order_items_df = pd.read_csv(RAW_DATA_DIR / "order_items.csv")
customers_df = pd.read_csv(RAW_DATA_DIR / "customers.csv")
product_variants_df = pd.read_csv(RAW_DATA_DIR / "product_variants.csv")
products_df = pd.read_csv(RAW_DATA_DIR / "products.csv")
categories_df = pd.read_csv(RAW_DATA_DIR / "categories.csv")
brands_df = pd.read_csv(RAW_DATA_DIR / "brands.csv")
locations_df = pd.read_csv(RAW_DATA_DIR / "locations.csv")
employees_df = pd.read_csv(RAW_DATA_DIR / "employees.csv")
returns_df = pd.read_csv(RAW_DATA_DIR / "returns.csv")
return_items_df = pd.read_csv(RAW_DATA_DIR / "return_items.csv")

print("Fontes carregadas: 11")

Fontes carregadas: 11


In [31]:
source_validation = [
    ("orders.csv", orders_df, "id", 48_998),
    ("order_items.csv", order_items_df, "id", 147_320),
    ("customers.csv", customers_df, "id", 2_000),
    ("product_variants.csv", product_variants_df, "id", 1_009),
    ("products.csv", products_df, "id", None),
    ("categories.csv", categories_df, "id", None),
    ("brands.csv", brands_df, "id", None),
    ("locations.csv", locations_df, "id", 6),
    ("employees.csv", employees_df, "id", 15),
    ("returns.csv", returns_df, "id", None),
    ("return_items.csv", return_items_df, "id", None),
]

# 6. Validação inicial

O resumo verifica dimensões, duplicidades na chave esperada, total de valores nulos e memória profunda utilizada. As contagens informadas no desafio são comparadas sem modificar os DataFrames.

In [32]:
validation_records = []

for file_name, dataframe, expected_key, expected_rows in source_validation:
    actual_rows = len(dataframe)

    validation_records.append(
        {
            "arquivo": file_name,
            "linhas": actual_rows,
            "linhas_esperadas": expected_rows,
            "contagem_confere": expected_rows is None or actual_rows == expected_rows,
            "colunas": dataframe.shape[1],
            "chave_esperada": expected_key,
            "duplicidades_chave": int(dataframe.duplicated(subset=[expected_key]).sum()),
            "nulos": int(dataframe.isna().sum().sum()),
            "memoria_bytes": int(dataframe.memory_usage(index=True, deep=True).sum()),
        }
    )

validation_summary = pd.DataFrame(validation_records)
validation_summary["memoria_mib"] = (
    validation_summary["memoria_bytes"] / (1024 ** 2)
).round(2)

print(validation_summary.to_string(index=False))

             arquivo  linhas  linhas_esperadas  contagem_confere  colunas chave_esperada  duplicidades_chave  nulos  memoria_bytes  memoria_mib
          orders.csv   48998           48998.0              True       13             id                   0  24131       20995699        20.02
     order_items.csv  147320          147320.0              True        8             id                   0      0        9428612         8.99
       customers.csv    2000            2000.0              True       11             id                   0   2664         993112         0.95
product_variants.csv    1009            1009.0              True       12             id                   0    157         262472         0.25
        products.csv     500               NaN              True       10             id                   0     10         195542         0.19
      categories.csv      14               NaN              True        7             id                   0      3           4073      

In [33]:
for file_name, dataframe, _, _ in source_validation:
    columns = dataframe.columns.tolist()
    print(f"{file_name} ({len(columns)} colunas): {', '.join(columns)}")

orders.csv (13 colunas): id, order_number, channel, customer_id, salesperson_id, location_id, status, subtotal, discount_amount, total, placed_at, created_at, updated_at
order_items.csv (8 colunas): id, order_id, product_variant_id, quantity, unit_price, icms_rate, ipi_rate, line_total
customers.csv (11 colunas): id, person_type, legal_name, trade_name, tax_id, state_registration, email, phone, is_active, created_at, updated_at
product_variants.csv (12 colunas): id, product_id, sku, barcode_ean, sale_price, cost_price, weight_kg, icms_rate, ipi_rate, is_active, created_at, updated_at
products.csv (10 colunas): id, name, description, brand_id, category_id, ncm_code, unit_of_measure, is_active, created_at, updated_at
categories.csv (7 colunas): id, name, slug, parent_category_id, is_active, created_at, updated_at
brands.csv (6 colunas): id, name, country, is_active, created_at, updated_at
locations.csv (14 colunas): id, name, location_type, postal_code, street, number, complement, distri

# 7. Dimensões

Construção das quatro dimensões Gold em memória. Nenhum arquivo é exportado nesta etapa.

## 7.1 Dimensão de datas

Calendário diário contínuo de 01/01/2020 a 31/12/2027. `day_of_week` segue a convenção pandas (segunda-feira = 0), enquanto `iso_day_of_week` segue a ISO 8601 (segunda-feira = 1). Os nomes são definidos por mapas explícitos para não depender do locale do ambiente.

In [34]:
DAY_NAMES_PT = {
    1: "segunda-feira", 2: "terça-feira", 3: "quarta-feira",
    4: "quinta-feira", 5: "sexta-feira", 6: "sábado", 7: "domingo",
}
MONTH_NAMES_PT = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro",
}

calendar_dates = pd.date_range(start="2020-01-01", end="2027-12-31", freq="D")
iso_calendar = calendar_dates.to_series(index=range(len(calendar_dates))).dt.isocalendar()

dim_date = pd.DataFrame({"date": calendar_dates})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype("int32")
dim_date["year"] = dim_date["date"].dt.year
dim_date["day_of_year"] = dim_date["date"].dt.dayofyear
dim_date["day_of_month"] = dim_date["date"].dt.day
dim_date["iso_day_of_week"] = iso_calendar["day"].astype("int8").to_numpy()
dim_date["day_name_pt"] = dim_date["iso_day_of_week"].map(DAY_NAMES_PT)
dim_date["week_of_year"] = iso_calendar["week"].astype("int16").to_numpy()
dim_date["month"] = dim_date["date"].dt.month
dim_date["year_month"] = dim_date["date"].dt.strftime("%Y-%m")
dim_date["month_name"] = dim_date["month"].map(MONTH_NAMES_PT)
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["quarter_name"] = "Q" + dim_date["quarter"].astype(str)
dim_date["semester"] = ((dim_date["month"] - 1) // 6 + 1).astype("int8")
dim_date["trimestre"] = "T" + dim_date["quarter"].astype(str)
dim_date["is_weekend"] = dim_date["iso_day_of_week"].isin([6, 7])

dim_date.head(2)

,date,date_key,year,day_of_year,day_of_month,iso_day_of_week,day_name_pt,week_of_year,month,year_month,month_name,quarter,quarter_name,semester,trimestre,is_weekend
0,2020-01-01,20200101,2020,1,1,3,quarta-feira,1,1,2020-01,Janeiro,1,Q1,1,T1,False
1,2020-01-02,20200102,2020,2,2,4,quinta-feira,1,1,2020-01,Janeiro,1,Q1,1,T1,False


## 7.2 Dimensão de clientes

Uma linha por cliente, contendo somente a chave e atributos descritivos da fonte.

In [35]:
dim_customer = (
    customers_df
    .rename(columns={"id": "customer_id"})
    .drop(columns=["created_at", "updated_at"])
)

dim_customer.head(2)

,customer_id,person_type,legal_name,trade_name,tax_id,state_registration,email,phone,is_active
0,1,PJ,Alves e Filhos S.A.,da Cruz,11713629000103,321819600,alves.e.filhos.s.a.1@example.com,5533989083863,True
1,2,PF,Lara Abreu,NaN,72890168662,NaN,lara.abreu.2@example.com,55065942351161,False


## 7.3 Dimensão de produtos

Uma linha por variante. Os joins são feitos à esquerda a partir de `product_variants`, preservando variantes ativas e inativas e validando cardinalidade muitos-para-um.

In [36]:
variant_attributes = (
    product_variants_df
    .rename(columns={
        "id": "product_variant_id",
        "is_active": "is_active_variant",
    })
    .drop(columns=["created_at", "updated_at"])
)

product_attributes = (
    products_df
    .rename(columns={
        "id": "product_id",
        "name": "product_name",
        "is_active": "is_active_product",
    })
    .drop(columns=["created_at", "updated_at"])
)

category_attributes = (
    categories_df
    .rename(columns={
        "id": "category_id",
        "name": "category_name",
        "is_active": "is_active_category",
    })
    .drop(columns=["created_at", "updated_at"])
)

brand_attributes = (
    brands_df
    .rename(columns={
        "id": "brand_id",
        "name": "brand_name",
        "is_active": "is_active_brand",
    })
    .drop(columns=["created_at", "updated_at"])
)

product_dimension_join = (
    variant_attributes
    .merge(
        product_attributes, on="product_id", how="left", validate="many_to_one",
        indicator="_product_join",
    )
    .merge(
        category_attributes, on="category_id", how="left", validate="many_to_one",
        indicator="_category_join",
    )
    .merge(
        brand_attributes, on="brand_id", how="left", validate="many_to_one",
        indicator="_brand_join",
    )
)

dim_product = product_dimension_join.drop(
    columns=["_product_join", "_category_join", "_brand_join"]
)

dim_product.head(2)

,product_variant_id,product_id,sku,barcode_ean,sale_price,cost_price,weight_kg,icms_rate,ipi_rate,is_active_variant,...,ncm_code,unit_of_measure,is_active_product,category_name,slug,parent_category_id,is_active_category,brand_name,country,is_active_brand
0,1,1,LHN-353137,8.123564e+11,2452.69,1409.59,17.466,18.0,5.0,True,...,50675645,UN,True,Acessórios de Convés,acessorios-de-conves,2.0,True,Volvo Penta,JP,True
1,2,1,LHN-424236,2.876605e+12,1057.68,482.96,38.900,18.0,10.0,True,...,50675645,UN,True,Acessórios de Convés,acessorios-de-conves,2.0,True,Volvo Penta,JP,True


## 7.4 Dimensão de localizações

Uma linha por localização, com nome, tipo e atributos geográficos úteis.

In [37]:
dim_location = (
    locations_df
    .rename(columns={"id": "location_id"})
    .drop(columns=["created_at", "updated_at"])
)

dim_location.head(2)

,location_id,name,location_type,postal_code,street,number,complement,district,city,state,country,is_active
0,1,Loja Leão,store,64771-158,Conjunto de Siqueira,9,Sala 70,Vila Independencia 2ª Seção,Barbosa do Sul,PR,BR,True
1,2,Armazém Camargo,warehouse,11464-461,Trecho Montenegro,80,NaN,Marieta 1ª Seção,Cavalcanti da Mata,PR,BR,True


## 7.5 Dimensão de funcionários

Uma linha por funcionário, com atributos descritivos para análise e ranking de vendedores. Funcionários desligados e inativos são preservados.

In [38]:
dim_employee = (
    employees_df
    .rename(columns={"id": "employee_id"})
    .drop(columns=["created_at", "updated_at"])
)

dim_employee.head(2)

,employee_id,full_name,cpf,email,role,primary_location_id,hire_date,termination_date,is_active
0,1,Maria Luísa Rodrigues,42679135105,maria.lu.sa.rodrigues.1@lhnautical.com.br,manager,5,2022-03-04,NaN,True
1,2,Nicolas Ferreira,91124322450,nicolas.ferreira.2@lhnautical.com.br,stockist,4,2025-02-19,NaN,True


## 7.6 Validação das dimensões

As assertions interrompem a execução se houver contagens inesperadas, chaves repetidas ou nulas, falhas na continuidade do calendário ou perda de integridade nos joins da dimensão de produtos.

In [39]:
EXPECTED_DIMENSION_COUNTS = {
    "dim_date": 2_922,
    "dim_customer": 2_000,
    "dim_product": 1_009,
    "dim_location": 6,
    "dim_employee": 15,
}
DIMENSION_KEYS = {
    "dim_date": "date_key",
    "dim_customer": "customer_id",
    "dim_product": "product_variant_id",
    "dim_location": "location_id",
    "dim_employee": "employee_id",
}
gold_dimensions = {
    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_location": dim_location,
    "dim_employee": dim_employee,
}

for dimension_name, dataframe in gold_dimensions.items():
    dimension_key = DIMENSION_KEYS[dimension_name]
    assert dataframe[dimension_key].notna().all(), f"{dimension_name}: chave nula encontrada."
    assert dataframe[dimension_key].is_unique, f"{dimension_name}: chave duplicada encontrada."

assert dim_date["date"].min() == pd.Timestamp("2020-01-01")
assert dim_date["date"].max() == pd.Timestamp("2027-12-31")
assert dim_date["date"].diff().dropna().eq(pd.Timedelta(days=1)).all(), (
    "dim_date: o calendário contém lacunas."
)

for join_column in ["_product_join", "_category_join", "_brand_join"]:
    assert product_dimension_join[join_column].eq("both").all(), (
        f"dim_product: integridade violada em {join_column}."
    )

assert len(product_dimension_join) == len(product_variants_df), (
    "dim_product: os joins alteraram a quantidade de variantes."
)
assert len(dim_employee) == len(employees_df), (
    "dim_employee: a transformação alterou a quantidade de funcionários."
)
assert dim_employee["primary_location_id"].isin(dim_location["location_id"]).all(), (
    "dim_employee: existe localização principal sem correspondência."
)

print("Assertions das cinco dimensões concluídas sem erros.")

Assertions das cinco dimensões concluídas sem erros.


In [40]:
for dimension_name, dataframe in gold_dimensions.items():
    print(f"\n{'=' * 100}\n{dimension_name}: amostra")
    print(dataframe.head(3).to_string(index=False))
    print(f"\n{dimension_name}: estrutura")
    dataframe.info(memory_usage="deep")


dim_date: amostra
      date  date_key  year  day_of_year  day_of_month  iso_day_of_week  day_name_pt  week_of_year  month year_month month_name  quarter quarter_name  semester trimestre  is_weekend
2020-01-01  20200101  2020            1             1                3 quarta-feira             1      1    2020-01    Janeiro        1           Q1         1        T1       False
2020-01-02  20200102  2020            2             2                4 quinta-feira             1      1    2020-01    Janeiro        1           Q1         1        T1       False
2020-01-03  20200103  2020            3             3                5  sexta-feira             1      1    2020-01    Janeiro        1           Q1         1        T1       False

dim_date: estrutura
<class 'pandas.DataFrame'>
RangeIndex: 2922 entries, 0 to 2921
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             2922 non-nul

# 8. Fatos

Construção das quatro tabelas fato Gold em memória. Os registros das fontes são preservados na granularidade original e nenhum arquivo é exportado nesta etapa.


## 8.1 Fato de pedidos

Uma linha por pedido. `placed_date` e `date_key` são derivados exclusivamente de `placed_at`; `created_at` não participa dos indicadores comerciais.

In [41]:
fact_orders = (
    orders_df
    .rename(columns={"id": "order_id"})
    .drop(columns=["created_at", "updated_at"])
    .copy()
)

placed_datetime = pd.to_datetime(fact_orders["placed_at"])
fact_orders["date_key"] = placed_datetime.dt.strftime("%Y%m%d").astype("int32")

fact_orders.head(2)

,order_id,order_number,channel,customer_id,salesperson_id,location_id,status,subtotal,discount_amount,total,placed_at,date_key
0,1,SO-000001,ecommerce,1136,NaN,1,paid,323.34,35.57,287.77,2022-09-06 05:37:37,20220906
1,2,SO-000002,ecommerce,618,9.0,4,paid,53199.05,0.00,53199.05,2023-02-03 04:36:21,20230203


## 8.2 Fato de itens de pedido

Uma linha por item. A fato mantém somente os atributos já existentes na granularidade do item e não recebe valores monetários do cabeçalho do pedido.

In [42]:
fact_order_items = order_items_df.rename(columns={"id": "order_item_id"})

fact_order_items.head(2)

,order_item_id,order_id,product_variant_id,quantity,unit_price,icms_rate,ipi_rate,line_total
0,1,1,113,6,53.89,12.0,0.0,323.34
1,2,2,293,9,2398.41,17.0,10.0,21585.69


## 8.3 Fato de devoluções

Uma linha por devolução registrada em `returns.csv`. A granularidade e os atributos da fonte são preservados; a chave `id` é renomeada para `return_id` para explicitar seu papel no modelo Gold.


In [43]:
fact_returns = returns_df.rename(columns={"id": "return_id"}).copy()

fact_returns.head(2)


,return_id,return_number,order_id,customer_id,received_at_location_id,status,reason,total_refund_amount,created_at,updated_at
0,1,RT-000001,42945,484,3,cancelled,Produto avariado no transporte,6004.35,2026-04-25 22:45:38,2026-04-25 22:45:38
1,2,RT-000002,48935,741,1,completed,Item não corresponde à descrição,2277.61,2025-01-08 10:45:17,2025-01-08 10:45:17


## 8.4 Fato de itens devolvidos

Uma linha por item de devolução registrado em `return_items.csv`. A granularidade e os atributos da fonte são preservados; a chave `id` é renomeada para `return_item_id`.


In [44]:
fact_return_items = return_items_df.rename(columns={"id": "return_item_id"}).copy()

fact_return_items.head(2)


,return_item_id,return_id,order_item_id,quantity,action,exchange_variant_id,unit_refund_amount
0,1,1,129232,5.0,refund,NaN,1200.87
1,2,2,147166,1.0,refund,NaN,2277.61


## 8.5 Validação das tabelas fato

As validações confirmam granularidade, chaves, integridade referencial quando as respectivas FKs existem, reconciliação monetária dos pedidos e ausência de replicação do faturamento dos pedidos nos itens.


In [45]:
orders_without_customer = (
    fact_orders["customer_id"].isna()
    | ~fact_orders["customer_id"].isin(dim_customer["customer_id"])
).sum()
orders_without_location = (
    fact_orders["location_id"].isna()
    | ~fact_orders["location_id"].isin(dim_location["location_id"])
).sum()
orders_with_unknown_salesperson = (
    fact_orders["salesperson_id"].notna()
    & ~fact_orders["salesperson_id"].isin(dim_employee["employee_id"])
).sum()
items_without_order = (
    fact_order_items["order_id"].isna()
    | ~fact_order_items["order_id"].isin(fact_orders["order_id"])
).sum()
items_without_variant = (
    fact_order_items["product_variant_id"].isna()
    | ~fact_order_items["product_variant_id"].isin(dim_product["product_variant_id"])
).sum()

returns_without_order = None
if "order_id" in fact_returns.columns:
    returns_without_order = (
        fact_returns["order_id"].isna()
        | ~fact_returns["order_id"].isin(fact_orders["order_id"])
    ).sum()

returns_without_customer = None
if "customer_id" in fact_returns.columns:
    returns_without_customer = (
        fact_returns["customer_id"].notna()
        & ~fact_returns["customer_id"].isin(dim_customer["customer_id"])
    ).sum()

return_items_without_return = None
if "return_id" in fact_return_items.columns:
    return_items_without_return = (
        fact_return_items["return_id"].isna()
        | ~fact_return_items["return_id"].isin(fact_returns["return_id"])
    ).sum()

return_items_without_order_item = None
if "order_item_id" in fact_return_items.columns:
    return_items_without_order_item = (
        fact_return_items["order_item_id"].isna()
        | ~fact_return_items["order_item_id"].isin(fact_order_items["order_item_id"])
    ).sum()

return_items_without_variant = None
if "product_variant_id" in fact_return_items.columns:
    return_items_without_variant = (
        fact_return_items["product_variant_id"].notna()
        & ~fact_return_items["product_variant_id"].isin(dim_product["product_variant_id"])
    ).sum()

order_total_difference = (
    fact_orders["subtotal"] - fact_orders["discount_amount"] - fact_orders["total"]
).abs()
incompatible_order_totals = order_total_difference.gt(0.01).sum()

assert len(fact_orders) == len(orders_df), "fact_orders: a transformação alterou a quantidade de pedidos."
assert len(fact_order_items) == len(order_items_df), (
    "fact_order_items: a transformação alterou a quantidade de itens."
)
assert len(fact_returns) == len(returns_df), (
    "fact_returns: a transformação alterou a quantidade de devoluções."
)
assert len(fact_return_items) == len(return_items_df), (
    "fact_return_items: a transformação alterou a quantidade de itens devolvidos."
)
assert fact_orders["order_id"].notna().all(), "fact_orders: chave nula encontrada."
assert fact_orders["order_id"].is_unique, "fact_orders: chave duplicada encontrada."
assert fact_order_items["order_item_id"].notna().all(), (
    "fact_order_items: chave nula encontrada."
)
assert fact_order_items["order_item_id"].is_unique, (
    "fact_order_items: chave duplicada encontrada."
)
assert fact_returns["return_id"].notna().all(), (
    "fact_returns: chave nula encontrada."
)
assert fact_returns["return_id"].is_unique, (
    "fact_returns: chave duplicada encontrada."
)
assert fact_return_items["return_item_id"].notna().all(), (
    "fact_return_items: chave nula encontrada."
)
assert fact_return_items["return_item_id"].is_unique, (
    "fact_return_items: chave duplicada encontrada."
)
if returns_without_order is not None:
    assert returns_without_order == 0, f"Devoluções sem pedido válido: {returns_without_order}."
if returns_without_customer is not None:
    assert returns_without_customer == 0, f"Devoluções com cliente sem correspondência: {returns_without_customer}."
if return_items_without_return is not None:
    assert return_items_without_return == 0, (
        f"Itens devolvidos sem devolução válida: {return_items_without_return}."
    )
if return_items_without_order_item is not None:
    assert return_items_without_order_item == 0, (
        f"Itens devolvidos sem item de pedido válido: {return_items_without_order_item}."
    )
if return_items_without_variant is not None:
    assert return_items_without_variant == 0, (
        f"Itens devolvidos com variante sem correspondência: {return_items_without_variant}."
    )
assert orders_without_customer == 0, f"Pedidos sem cliente válido: {orders_without_customer}."
assert orders_without_location == 0, f"Pedidos sem local válido: {orders_without_location}."
assert orders_with_unknown_salesperson == 0, (
    f"Pedidos com vendedor sem correspondência: {orders_with_unknown_salesperson}."
)
assert items_without_order == 0, f"Itens sem pedido válido: {items_without_order}."
assert items_without_variant == 0, f"Itens sem variante válida: {items_without_variant}."
assert incompatible_order_totals == 0, (
    f"Pedidos incompatíveis com subtotal - desconto = total: {incompatible_order_totals}."
)
assert fact_orders["date_key"].isin(dim_date["date_key"]).all(), (
    "fact_orders: existe date_key fora da dimensão de datas."
)
assert fact_orders["placed_at"].equals(orders_df["placed_at"]), (
    "fact_orders: placed_at foi alterado em relação à fonte."
)
assert {"order_number", "salesperson_id"}.issubset(fact_orders.columns), (
    "fact_orders: order_number ou salesperson_id não foi preservado."
)
assert {"total", "subtotal", "discount_amount"}.isdisjoint(fact_order_items.columns), (
    "fact_order_items: valores do cabeçalho do pedido foram replicados nos itens."
)
assert abs(fact_orders["total"].sum() - orders_df["total"].sum()) <= 0.01, (
    "fact_orders: o faturamento total foi alterado ou duplicado."
)

print("Validações das tabelas fato concluídas sem erros.")
print(f"Pedidos sem cliente válido: {orders_without_customer}")
print(f"Pedidos sem local válido: {orders_without_location}")
print(f"Pedidos com vendedor sem correspondência: {orders_with_unknown_salesperson}")
print(f"Pedidos sem vendedor informado: {fact_orders['salesperson_id'].isna().sum()}")
print(f"Itens sem pedido válido: {items_without_order}")
print(f"Itens sem variante válida: {items_without_variant}")
print(f"Pedidos com total incompatível: {incompatible_order_totals}")
print(f"Faturamento preservado: {fact_orders['total'].sum():,.2f}")
print(f"Status preservados: {sorted(fact_orders['status'].unique().tolist())}")

Validações das tabelas fato concluídas sem erros.
Pedidos sem cliente válido: 0
Pedidos sem local válido: 0
Pedidos com vendedor sem correspondência: 0
Pedidos sem vendedor informado: 24131
Itens sem pedido válido: 0
Itens sem variante válida: 0
Pedidos com total incompatível: 0
Faturamento preservado: 1,406,487,201.80
Status preservados: ['cancelled', 'confirmed', 'draft', 'paid']


In [46]:
for fact_name, dataframe in [
    ("fact_orders", fact_orders),
    ("fact_order_items", fact_order_items),
    ("fact_returns", fact_returns),
    ("fact_return_items", fact_return_items),
]:
    print(f"\n{'=' * 100}\n{fact_name}: amostra")
    print(dataframe.head(3).to_string(index=False))
    print(f"\n{fact_name}: estrutura")
    dataframe.info(memory_usage="deep")



fact_orders: amostra
 order_id order_number   channel  customer_id  salesperson_id  location_id    status  subtotal  discount_amount    total           placed_at  date_key
        1    SO-000001 ecommerce         1136             NaN            1      paid    323.34            35.57   287.77 2022-09-06 05:37:37  20220906
        2    SO-000002 ecommerce          618             9.0            4      paid  53199.05             0.00 53199.05 2023-02-03 04:36:21  20230203
        3    SO-000003       pos          227            10.0            4 confirmed  17157.39             0.00 17157.39 2024-12-30 07:15:17  20241230

fact_orders: estrutura
<class 'pandas.DataFrame'>
RangeIndex: 48998 entries, 0 to 48997
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         48998 non-null  int64  
 1   order_number     48998 non-null  str    
 2   channel          48998 non-null  str    
 3   customer_id      

# 9. Clientes fiéis

Resultados Gold em memória para identificar clientes fiéis e as categorias mais consumidas pelo grupo. Somente pedidos pagos são considerados e nenhum arquivo é exportado nesta etapa.

## 9.1 Métricas de pedidos e diversidade

Faturamento e quantidade de pedidos são calculados diretamente na granularidade de pedidos. A diversidade de categorias é calculada separadamente nos itens, evitando replicar `fact_orders.total`.

In [47]:
paid_orders = fact_orders[fact_orders["status"].eq("paid")]

paid_order_metrics = (
    paid_orders
    .groupby("customer_id", as_index=False)
    .agg(
        paid_orders=("order_id", "nunique"),
        paid_revenue=("total", "sum"),
    )
)
paid_order_metrics["average_ticket"] = (
    paid_order_metrics["paid_revenue"] / paid_order_metrics["paid_orders"]
)

paid_customer_items = (
    fact_order_items
    .merge(
        paid_orders[["order_id", "customer_id"]],
        on="order_id",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        dim_product[["product_variant_id", "category_id"]],
        on="product_variant_id",
        how="left",
        validate="many_to_one",
    )
)

customer_category_diversity = (
    paid_customer_items
    .groupby("customer_id", as_index=False)
    .agg(category_diversity=("category_id", "nunique"))
)

loyal_customer_metrics = paid_order_metrics.merge(
    customer_category_diversity,
    on="customer_id",
    how="left",
    validate="one_to_one",
)

## 9.2 Top 10 de clientes fiéis

Clientes elegíveis possuem pelo menos 13 categorias distintas. O ranking usa ticket médio decrescente e `customer_id` crescente como desempate.

In [48]:
eligible_loyal_customers = loyal_customer_metrics[
    loyal_customer_metrics["category_diversity"].ge(13)
]

aux_loyal_customers = (
    eligible_loyal_customers
    .merge(dim_customer, on="customer_id", how="left", validate="one_to_one")
    .sort_values(
        ["average_ticket", "customer_id"],
        ascending=[False, True],
        kind="stable",
    )
    .head(10)
    .reset_index(drop=True)
)

aux_loyal_customers["ranking_position"] = range(1, len(aux_loyal_customers) + 1)

loyal_customer_decimal_columns = [
    column
    for column in aux_loyal_customers.columns
    if pd.api.types.is_float_dtype(aux_loyal_customers[column])
]
aux_loyal_customers[loyal_customer_decimal_columns] = (
    aux_loyal_customers[loyal_customer_decimal_columns].round(2)
)

aux_loyal_customers.head(10)

,customer_id,paid_orders,paid_revenue,average_ticket,category_diversity,person_type,legal_name,trade_name,tax_id,state_registration,email,phone,is_active,ranking_position
0,1527,15,699704.65,46646.98,13,PF,Caio Farias,NaN,8581402500,NaN,caio.farias.1527@example.com,5554923324165,True,1
1,1581,12,534022.77,44501.90,14,PJ,Novaes Andrade S.A.,Ramos - EI,14481123000104,893063578,novaes.andrade.s.a.1581@example.com,5537989121397,True,2
2,1558,11,487829.66,44348.15,13,PF,Natália Borges,NaN,70316451070,NaN,nat.lia.borges.1558@example.com,5567906833539,True,3
3,262,12,525854.13,43821.18,14,PJ,Castro Cirino S.A.,Correia,24305515000168,129087814,castro.cirino.s.a.262@example.com,5541939755231,True,4
4,22,18,777241.65,43180.09,14,PF,Isadora Rios,NaN,98235269740,NaN,isadora.rios.22@example.com,5558969232260,True,5
5,1470,21,901262.24,42917.25,14,PF,Bárbara Albuquerque,NaN,19017412577,NaN,b.rbara.albuquerque.1470@example.com,5590991203025,True,6
6,1113,15,642404.60,42826.97,14,PF,Melissa Albuquerque,NaN,55094818970,NaN,melissa.albuquerque.1113@example.com,5542962037087,True,7
7,1116,12,502685.60,41890.47,13,PF,Thomas Alves,NaN,75163536800,NaN,thomas.alves.1116@example.com,5586944822175,True,8
8,1301,14,586202.80,41871.63,13,PJ,Correia Campos - EI S.A.,NaN,77393395000164,361133892,correia.campos.ei.s.a.1301@example.com,5508978828152,True,9
9,1814,20,835340.43,41767.02,14,PJ,Viana ME,Andrade,86146861000106,977779570,viana.me.1814@example.com,5599962637792,False,10


## 9.3 Categorias consumidas pelo Top 10

A quantidade é somada apenas para itens de pedidos pagos pertencentes aos dez clientes selecionados. Em caso de empate, `category_id` crescente garante ordenação determinística.

In [49]:
top_10_customer_ids = aux_loyal_customers["customer_id"]
loyal_customer_items = paid_customer_items[
    paid_customer_items["customer_id"].isin(top_10_customer_ids)
]

category_lookup = (
    dim_product[["category_id", "category_name"]]
    .drop_duplicates()
)

aux_loyal_categories = (
    loyal_customer_items
    .groupby("category_id", as_index=False)
    .agg(total_quantity=("quantity", "sum"))
    .merge(category_lookup, on="category_id", how="left", validate="one_to_one")
    .sort_values(
        ["total_quantity", "category_id"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)
aux_loyal_categories["ranking_position"] = range(1, len(aux_loyal_categories) + 1)

loyal_category_decimal_columns = [
    column
    for column in aux_loyal_categories.columns
    if pd.api.types.is_float_dtype(aux_loyal_categories[column])
]
aux_loyal_categories[loyal_category_decimal_columns] = (
    aux_loyal_categories[loyal_category_decimal_columns].round(2)
)


## 9.4 Validação dos clientes fiéis

As assertions comparam os resultados calculados aos valores de referência sem substituir ou forçar qualquer valor.

In [50]:
expected_top_10_customer_ids = [
    1527, 1581, 1558, 262, 22, 1470, 1113, 1116, 1301, 1814,
]
actual_top_10_customer_ids = aux_loyal_customers["customer_id"].tolist()

assert len(eligible_loyal_customers) == 1_775, (
    f"Clientes elegíveis: esperados 1775, encontrados {len(eligible_loyal_customers)}."
)
assert actual_top_10_customer_ids == expected_top_10_customer_ids, (
    f"Top 10 divergente: {actual_top_10_customer_ids}."
)
assert aux_loyal_customers["customer_id"].is_unique
assert aux_loyal_customers["category_diversity"].ge(13).all()
assert len(paid_customer_items) == fact_order_items[
    fact_order_items["order_id"].isin(paid_orders["order_id"])
].shape[0], "Os joins alteraram a quantidade de itens pagos."
assert abs(paid_order_metrics["paid_revenue"].sum() - paid_orders["total"].sum()) <= 0.01, (
    "O faturamento pago foi duplicado durante a agregação por cliente."
)
assert aux_loyal_categories.iloc[0]["category_name"] == "Equipamentos", (
    f"Categoria líder divergente: {aux_loyal_categories.iloc[0]['category_name']}."
)
assert aux_loyal_categories.iloc[0]["total_quantity"] == 291, (
    f"Quantidade líder divergente: {aux_loyal_categories.iloc[0]['total_quantity']}."
)


for dataframe_name, dataframe, decimal_columns in [
    ("aux_loyal_customers", aux_loyal_customers, loyal_customer_decimal_columns),
    ("aux_loyal_categories", aux_loyal_categories, loyal_category_decimal_columns),
]:
    for column in decimal_columns:
        values = dataframe[column].dropna()
        assert values.eq(values.round(2)).all(), (
            f"{dataframe_name}.{column}: encontrado valor com mais de duas casas decimais."
        )

print("Validações de clientes fiéis concluídas sem erros.")
print(f"Clientes elegíveis: {len(eligible_loyal_customers)}")
print(f"Top 10: {actual_top_10_customer_ids}")
print(
    f"Categoria líder: {aux_loyal_categories.iloc[0]['category_name']} "
    f"({aux_loyal_categories.iloc[0]['total_quantity']} unidades)"
)

Validações de clientes fiéis concluídas sem erros.
Clientes elegíveis: 1775
Top 10: [1527, 1581, 1558, 262, 22, 1470, 1113, 1116, 1301, 1814]
Categoria líder: Equipamentos (291 unidades)


In [51]:
print("aux_loyal_customers")
print(aux_loyal_customers.to_string(index=False))
print("\naux_loyal_categories")
print(aux_loyal_categories.to_string(index=False))

aux_loyal_customers
 customer_id  paid_orders  paid_revenue  average_ticket  category_diversity person_type               legal_name trade_name         tax_id state_registration                                  email         phone  is_active  ranking_position
        1527           15     699704.65        46646.98                  13          PF              Caio Farias        NaN     8581402500                NaN           caio.farias.1527@example.com 5554923324165       True                 1
        1581           12     534022.77        44501.90                  14          PJ      Novaes Andrade S.A. Ramos - EI 14481123000104          893063578    novaes.andrade.s.a.1581@example.com 5537989121397       True                 2
        1558           11     487829.66        44348.15                  13          PF           Natália Borges        NaN    70316451070                NaN        nat.lia.borges.1558@example.com 5567906833539       True                 3
         262        

# 10. Recomendação

Resultado Gold em memória para recomendar os cinco produtos mais similares a `Motor de Popa 1949` a partir do histórico binário de compras pagas. Nenhum arquivo é exportado nesta etapa.

## 10.1 Interações cliente-produto

Pedidos pagos são unidos a itens, variantes e produtos. Compras repetidas do mesmo produto pelo mesmo cliente são reduzidas a uma única interação de valor 1; combinações sem compra recebem 0 na matriz.

In [52]:
REFERENCE_PRODUCT_NAME = "Motor de Popa 1949"

reference_product_matches = products_df[
    products_df["name"].eq(REFERENCE_PRODUCT_NAME)
]
assert not reference_product_matches.empty, (
    f"Produto de referência não encontrado: {REFERENCE_PRODUCT_NAME}."
)
assert len(reference_product_matches) == 1, (
    f"Produto de referência ambíguo: {len(reference_product_matches)} registros encontrados."
)
reference_product_id = int(reference_product_matches.iloc[0]["id"])

paid_recommendation_orders = (
    orders_df[orders_df["status"].eq("paid")]
    .rename(columns={"id": "order_id"})
)
recommendation_variants = product_variants_df.rename(
    columns={"id": "product_variant_id"}
)
recommendation_products = products_df.rename(
    columns={"id": "product_id", "name": "product_name"}
)

recommendation_join = (
    paid_recommendation_orders[["order_id", "customer_id"]]
    .merge(
        order_items_df[["order_id", "product_variant_id"]],
        on="order_id",
        how="inner",
        validate="one_to_many",
    )
    .merge(
        recommendation_variants[["product_variant_id", "product_id"]],
        on="product_variant_id",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        recommendation_products[["product_id", "product_name"]],
        on="product_id",
        how="inner",
        validate="many_to_one",
    )
)

customer_product_interactions = (
    recommendation_join[["customer_id", "product_id"]]
    .drop_duplicates()
)
customer_product_interactions["interaction"] = 1

customer_product_matrix = (
    customer_product_interactions
    .pivot(index="customer_id", columns="product_id", values="interaction")
    .fillna(0)
    .astype("int8")
)

## 10.2 Similaridade de cosseno

Para cada produto, a similaridade é o produto escalar entre seu vetor de clientes e o vetor do produto de referência, dividido pelo produto das normas dos dois vetores. A operação é calculada diretamente com pandas.

In [53]:
reference_vector = customer_product_matrix[reference_product_id]
product_dot_products = customer_product_matrix.T.dot(reference_vector)
product_norms = customer_product_matrix.pow(2).sum(axis=0).pow(0.5)
reference_norm = reference_vector.pow(2).sum() ** 0.5
product_similarities = product_dot_products / (product_norms * reference_norm)

recommendation_ranking = (
    product_similarities
    .drop(index=reference_product_id)
    .rename("similarity")
    .rename_axis("recommended_product_id")
    .reset_index()
    .sort_values(
        ["similarity", "recommended_product_id"],
        ascending=[False, True],
        kind="stable",
    )
    .head(5)
    .reset_index(drop=True)
)

recommended_product_lookup = (
    products_df
    .rename(columns={
        "id": "recommended_product_id",
        "name": "recommended_product_name",
    })
    .drop(columns=[
        "description", "brand_id", "category_id", "ncm_code",
        "unit_of_measure", "is_active", "created_at", "updated_at",
    ])
)

aux_recommendations = recommendation_ranking.merge(
    recommended_product_lookup,
    on="recommended_product_id",
    how="left",
    validate="one_to_one",
)
aux_recommendations.insert(0, "rank", range(1, len(aux_recommendations) + 1))
aux_recommendations.insert(1, "reference_product_id", reference_product_id)
aux_recommendations.insert(2, "reference_product_name", REFERENCE_PRODUCT_NAME)
aux_recommendations = aux_recommendations[[
    "rank", "reference_product_id", "reference_product_name",
    "recommended_product_id", "recommended_product_name", "similarity",
]]

### Observação de qualidade de dados

O nome de produto `asdf` aparenta ser um valor de teste ou cadastro inadequado. Como não há regra autorizando correção ou exclusão, ele é preservado sem renomeação e aparece normalmente no ranking calculado.

## 10.3 Validação das recomendações

As assertions verificam estrutura, ordenação e aderência aos valores de referência com tolerância absoluta de `1e-6`.

In [54]:
expected_recommendation_names = [
    "Vela Mestra 1913",
    "Cabo Náutico 2105",
    "Motor de Popa 6014",
    "asdf",
    "Âncora Bruce 7665",
]
expected_similarities = [0.204586, 0.189809, 0.187209, 0.185391, 0.185325]
similarity_tolerance = 1e-6

assert reference_product_id in customer_product_matrix.columns, (
    "Produto de referência ausente da matriz cliente-produto."
)
assert reference_product_id not in aux_recommendations["recommended_product_id"].tolist(), (
    "Produto de referência presente nas próprias recomendações."
)
assert len(aux_recommendations) == 5, (
    f"Quantidade de recomendações divergente: {len(aux_recommendations)}."
)
assert aux_recommendations["rank"].tolist() == [1, 2, 3, 4, 5]
assert aux_recommendations["similarity"].between(0, 1).all()
assert aux_recommendations["similarity"].is_monotonic_decreasing
assert set(customer_product_matrix.stack().unique().tolist()) == {0, 1}, (
    "A matriz cliente-produto contém valores diferentes de 0 e 1."
)
assert aux_recommendations["recommended_product_name"].tolist() == (
    expected_recommendation_names
), f"Produtos recomendados divergentes: {aux_recommendations['recommended_product_name'].tolist()}."
assert all(
    abs(actual - expected) <= similarity_tolerance
    for actual, expected in zip(
        aux_recommendations["similarity"], expected_similarities
    )
), f"Similaridades divergentes: {aux_recommendations['similarity'].tolist()}."

print("Validações das recomendações concluídas sem erros.")
print(
    f"Matriz cliente-produto: {customer_product_matrix.shape[0]} clientes x "
    f"{customer_product_matrix.shape[1]} produtos"
)
print(aux_recommendations.to_string(index=False))

Validações das recomendações concluídas sem erros.
Matriz cliente-produto: 2000 clientes x 500 produtos
 rank  reference_product_id reference_product_name  recommended_product_id recommended_product_name  similarity
    1                   180     Motor de Popa 1949                      75         Vela Mestra 1913    0.204586
    2                   180     Motor de Popa 1949                     295        Cabo Náutico 2105    0.189809
    3                   180     Motor de Popa 1949                       1       Motor de Popa 6014    0.187209
    4                   180     Motor de Popa 1949                     342                     asdf    0.185391
    5                   180     Motor de Popa 1949                     242        Âncora Bruce 7665    0.185325


# 12. Validação Gold

Validação consolidada antes da exportação. As assertions cobrem contagens, chaves, integridade referencial, calendário, reconciliação monetária e resultados analíticos.

In [55]:
paid_orders_gold = fact_orders[fact_orders["status"].eq("paid")]
paid_order_count = paid_orders_gold["order_id"].nunique()
paid_revenue = paid_orders_gold["total"].sum()
paid_average_ticket = paid_revenue / paid_order_count
paid_discount = paid_orders_gold["discount_amount"].sum()

gold_table_counts = [
    ("dim_date", dim_date, 2_922),
    ("dim_customer", dim_customer, 2_000),
    ("dim_product", dim_product, 1_009),
    ("dim_location", dim_location, 6),
    ("dim_employee", dim_employee, 15),
    ("fact_orders", fact_orders, 48_998),
    ("fact_order_items", fact_order_items, 147_320),
    ("fact_returns", fact_returns, len(returns_df)),
    ("fact_return_items", fact_return_items, len(return_items_df)),
    ("aux_loyal_customers", aux_loyal_customers, 10),
    ("aux_loyal_categories", aux_loyal_categories, 14),
    ("aux_recommendations", aux_recommendations, 5),
]
gold_table_keys = [
    ("dim_date", dim_date, "date_key"),
    ("dim_customer", dim_customer, "customer_id"),
    ("dim_product", dim_product, "product_variant_id"),
    ("dim_location", dim_location, "location_id"),
    ("dim_employee", dim_employee, "employee_id"),
    ("fact_orders", fact_orders, "order_id"),
    ("fact_order_items", fact_order_items, "order_item_id"),
    ("fact_returns", fact_returns, "return_id"),
    ("fact_return_items", fact_return_items, "return_item_id"),
    ("aux_loyal_customers", aux_loyal_customers, "customer_id"),
    ("aux_loyal_categories", aux_loyal_categories, "category_id"),
    ("aux_recommendations", aux_recommendations, "rank"),
]

for table_name, dataframe, expected_rows in gold_table_counts:
    assert len(dataframe) == expected_rows, (
        f"{table_name}: esperadas {expected_rows} linhas, encontradas {len(dataframe)}."
    )
    assert all(
        column == column.lower()
        and column.replace("_", "").isalnum()
        and not column.startswith("unnamed")
        for column in dataframe.columns
    ), f"{table_name}: existe coluna fora do padrão snake_case ou Unnamed."

for table_name, dataframe, key_column in gold_table_keys:
    assert dataframe[key_column].notna().all(), f"{table_name}: chave nula."
    assert dataframe[key_column].is_unique, f"{table_name}: chave duplicada."

assert fact_orders["customer_id"].isin(dim_customer["customer_id"]).all()
assert fact_orders["location_id"].isin(dim_location["location_id"]).all()
assert fact_orders["date_key"].isin(dim_date["date_key"]).all()
assert fact_orders.loc[
    fact_orders["salesperson_id"].notna(), "salesperson_id"
].isin(dim_employee["employee_id"]).all()
assert dim_employee["primary_location_id"].isin(dim_location["location_id"]).all()
assert fact_order_items["order_id"].isin(fact_orders["order_id"]).all()
assert fact_order_items["product_variant_id"].isin(
    dim_product["product_variant_id"]
).all()

if "order_id" in fact_returns.columns:
    assert fact_returns["order_id"].isin(fact_orders["order_id"]).all()
if "customer_id" in fact_returns.columns:
    assert fact_returns.loc[
        fact_returns["customer_id"].notna(), "customer_id"
    ].isin(dim_customer["customer_id"]).all()
if "return_id" in fact_return_items.columns:
    assert fact_return_items["return_id"].isin(fact_returns["return_id"]).all()
if "order_item_id" in fact_return_items.columns:
    assert fact_return_items["order_item_id"].isin(
        fact_order_items["order_item_id"]
    ).all()
if "product_variant_id" in fact_return_items.columns:
    assert fact_return_items.loc[
        fact_return_items["product_variant_id"].notna(), "product_variant_id"
    ].isin(dim_product["product_variant_id"]).all()

assert dim_date["date"].min() == pd.Timestamp("2020-01-01")
assert dim_date["date"].max() == pd.Timestamp("2027-12-31")
assert dim_date["date"].diff().dropna().eq(pd.Timedelta(days=1)).all()

gold_order_difference = (
    fact_orders["subtotal"] - fact_orders["discount_amount"] - fact_orders["total"]
).abs()
assert gold_order_difference.le(0.01).all(), (
    "Existem pedidos incompatíveis com subtotal - desconto = total."
)
assert fact_orders["order_id"].is_unique
assert abs(fact_orders["total"].sum() - orders_df["total"].sum()) <= 0.01
assert {"total", "subtotal", "discount_amount"}.isdisjoint(
    fact_order_items.columns
)

assert paid_order_count == 34_365
assert round(paid_revenue, 2) == 985_741_294.26
assert round(paid_average_ticket, 2) == 28_684.45
assert round(paid_discount, 2) == 21_692_622.12
assert len(eligible_loyal_customers) == 1_775
assert aux_loyal_categories.iloc[0]["category_name"] == "Equipamentos"
assert aux_loyal_categories.iloc[0]["total_quantity"] == 291
assert aux_recommendations["recommended_product_name"].tolist() == (
    expected_recommendation_names
)
assert all(
    abs(actual - expected) <= similarity_tolerance
    for actual, expected in zip(
        aux_recommendations["similarity"], expected_similarities
    )
)

print("Validação Gold pré-exportação concluída sem erros.")
print(f"Pedidos pagos: {paid_order_count:,}")
print(f"Receita paga: {paid_revenue:.2f}")
print(f"Ticket médio pago: {paid_average_ticket:.2f}")
print(f"Desconto pago: {paid_discount:.2f}")

Validação Gold pré-exportação concluída sem erros.
Pedidos pagos: 34,365
Receita paga: 985741294.26
Ticket médio pago: 28684.45
Desconto pago: 21692622.12


## 12.1 Observação sobre funcionários

`dim_employee` integra a exportação para permitir o relacionamento entre funcionários e os vendedores registrados em `fact_orders`. Pedidos sem vendedor permanecem nulos.

# 13. Exportação

Exportação determinística dos doze DataFrames Gold, com índice desabilitado, UTF-8 e datas em formato ISO. Duas escritas consecutivas são comparadas por hash para validar idempotência.


In [56]:
from hashlib import sha256

gold_export_spec = [
    ("dim_date.csv", dim_date, "date_key"),
    ("dim_customer.csv", dim_customer, "customer_id"),
    ("dim_product.csv", dim_product, "product_variant_id"),
    ("dim_location.csv", dim_location, "location_id"),
    ("dim_employee.csv", dim_employee, "employee_id"),
    ("fact_orders.csv", fact_orders, "order_id"),
    ("fact_order_items.csv", fact_order_items, "order_item_id"),
    ("fact_returns.csv", fact_returns, "return_id"),
    ("fact_return_items.csv", fact_return_items, "return_item_id"),
    ("aux_loyal_customers.csv", aux_loyal_customers, "ranking_position"),
    ("aux_loyal_categories.csv", aux_loyal_categories, "ranking_position"),
    ("aux_recommendations.csv", aux_recommendations, "rank"),
]
expected_gold_files = {file_name for file_name, _, _ in gold_export_spec}
GOLD_DATA_DIR.mkdir(parents=True, exist_ok=True)

for file_name, dataframe, sort_column in gold_export_spec:
    export_dataframe = dataframe.sort_values(sort_column, kind="stable").reset_index(drop=True)
    export_dataframe.to_csv(
        GOLD_DATA_DIR / file_name,
        index=False,
        encoding="utf-8",
        date_format="%Y-%m-%d",
    )

first_export_hashes = {
    file_name: sha256((GOLD_DATA_DIR / file_name).read_bytes()).hexdigest()
    for file_name in expected_gold_files
}

for file_name, dataframe, sort_column in gold_export_spec:
    export_dataframe = dataframe.sort_values(sort_column, kind="stable").reset_index(drop=True)
    export_dataframe.to_csv(
        GOLD_DATA_DIR / file_name,
        index=False,
        encoding="utf-8",
        date_format="%Y-%m-%d",
    )

second_export_hashes = {
    file_name: sha256((GOLD_DATA_DIR / file_name).read_bytes()).hexdigest()
    for file_name in expected_gold_files
}
assert first_export_hashes == second_export_hashes, (
    "A exportação não produziu conteúdo idempotente."
)
actual_gold_files = {path.name for path in GOLD_DATA_DIR.glob("*.csv")}
assert actual_gold_files == expected_gold_files, (
    f"Arquivos Gold divergentes. Esperados: {sorted(expected_gold_files)}; "
    f"encontrados: {sorted(actual_gold_files)}."
)
assert "aux_forecast_monthly.csv" not in actual_gold_files

print(f"Exportação concluída: {len(actual_gold_files)} arquivos CSV.")

Exportação concluída: 12 arquivos CSV.


## 13.1 Releitura e validação pós-exportação

Cada CSV é relido para validar conteúdo UTF-8, esquema, contagem, chave, tipos essenciais e relacionamentos.

In [57]:
expected_export_metadata = {
    "dim_date.csv": (dim_date, "date_key"),
    "dim_customer.csv": (dim_customer, "customer_id"),
    "dim_product.csv": (dim_product, "product_variant_id"),
    "dim_location.csv": (dim_location, "location_id"),
    "dim_employee.csv": (dim_employee, "employee_id"),
    "fact_orders.csv": (fact_orders, "order_id"),
    "fact_order_items.csv": (fact_order_items, "order_item_id"),
    "fact_returns.csv": (fact_returns, "return_id"),
    "fact_return_items.csv": (fact_return_items, "return_item_id"),
    "aux_loyal_customers.csv": (aux_loyal_customers, "customer_id"),
    "aux_loyal_categories.csv": (aux_loyal_categories, "category_id"),
    "aux_recommendations.csv": (aux_recommendations, "rank"),
}
essential_numeric_columns = {
    "dim_date.csv": ["date_key", "year", "month", "iso_day_of_week"],
    "dim_customer.csv": ["customer_id"],
    "dim_product.csv": ["product_variant_id", "product_id", "category_id"],
    "dim_location.csv": ["location_id"],
    "dim_employee.csv": ["employee_id", "primary_location_id"],
    "fact_orders.csv": ["order_id", "date_key", "salesperson_id", "total", "discount_amount"],
    "fact_order_items.csv": ["order_item_id", "order_id", "quantity", "line_total"],
    "fact_returns.csv": [],
    "fact_return_items.csv": [],
    "aux_loyal_customers.csv": ["customer_id", "paid_orders", "paid_revenue"],
    "aux_loyal_categories.csv": ["category_id", "total_quantity"],
    "aux_recommendations.csv": ["rank", "similarity"],
}
reloaded_gold = {}
gold_validation_records = []

for file_name, (source_dataframe, key_column) in expected_export_metadata.items():
    file_path = GOLD_DATA_DIR / file_name
    file_path.read_text(encoding="utf-8")
    reloaded_dataframe = pd.read_csv(file_path, encoding="utf-8")
    reloaded_gold[file_name] = reloaded_dataframe

    assert len(reloaded_dataframe) == len(source_dataframe), (
        f"{file_name}: perda ou aumento de linhas após exportação."
    )
    assert reloaded_dataframe.columns.tolist() == source_dataframe.columns.tolist(), (
        f"{file_name}: colunas divergentes após exportação."
    )
    assert not any(
        column.lower().startswith("unnamed") for column in reloaded_dataframe.columns
    ), f"{file_name}: coluna Unnamed encontrada."
    assert reloaded_dataframe[key_column].notna().all(), f"{file_name}: chave nula."
    assert reloaded_dataframe[key_column].is_unique, f"{file_name}: chave duplicada."
    assert all(
        pd.api.types.is_numeric_dtype(reloaded_dataframe[column])
        for column in essential_numeric_columns[file_name]
    ), f"{file_name}: perda de tipo numérico essencial."

    gold_validation_records.append({
        "path": str(file_path.resolve()),
        "size_bytes": file_path.stat().st_size,
        "rows": len(reloaded_dataframe),
        "columns": reloaded_dataframe.shape[1],
        "key": key_column,
        "status": "valid",
    })

assert reloaded_gold["dim_date.csv"]["date"].str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all(), "dim_date.csv: datas fora do padrão ISO."
assert reloaded_gold["fact_orders.csv"]["placed_at"].str.fullmatch(
    r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"
).all(), "fact_orders.csv: placed_at fora do padrão ISO."
assert reloaded_gold["dim_date.csv"]["is_weekend"].dtype == bool

assert reloaded_gold["fact_orders.csv"]["customer_id"].isin(
    reloaded_gold["dim_customer.csv"]["customer_id"]
).all()
assert reloaded_gold["fact_orders.csv"]["location_id"].isin(
    reloaded_gold["dim_location.csv"]["location_id"]
).all()
assert reloaded_gold["fact_orders.csv"]["date_key"].isin(
    reloaded_gold["dim_date.csv"]["date_key"]
).all()
reloaded_salesperson_ids = reloaded_gold["fact_orders.csv"].loc[
    reloaded_gold["fact_orders.csv"]["salesperson_id"].notna(),
    "salesperson_id",
]
assert reloaded_salesperson_ids.isin(
    reloaded_gold["dim_employee.csv"]["employee_id"]
).all()
assert reloaded_gold["dim_employee.csv"]["primary_location_id"].isin(
    reloaded_gold["dim_location.csv"]["location_id"]
).all()
assert reloaded_gold["fact_order_items.csv"]["order_id"].isin(
    reloaded_gold["fact_orders.csv"]["order_id"]
).all()
assert reloaded_gold["fact_order_items.csv"]["product_variant_id"].isin(
    reloaded_gold["dim_product.csv"]["product_variant_id"]
).all()

if "order_id" in reloaded_gold["fact_returns.csv"].columns:
    assert reloaded_gold["fact_returns.csv"]["order_id"].isin(
        reloaded_gold["fact_orders.csv"]["order_id"]
    ).all()
if "customer_id" in reloaded_gold["fact_returns.csv"].columns:
    reloaded_return_customer_ids = reloaded_gold["fact_returns.csv"].loc[
        reloaded_gold["fact_returns.csv"]["customer_id"].notna(), "customer_id"
    ]
    assert reloaded_return_customer_ids.isin(
        reloaded_gold["dim_customer.csv"]["customer_id"]
    ).all()
if "return_id" in reloaded_gold["fact_return_items.csv"].columns:
    assert reloaded_gold["fact_return_items.csv"]["return_id"].isin(
        reloaded_gold["fact_returns.csv"]["return_id"]
    ).all()
if "order_item_id" in reloaded_gold["fact_return_items.csv"].columns:
    assert reloaded_gold["fact_return_items.csv"]["order_item_id"].isin(
        reloaded_gold["fact_order_items.csv"]["order_item_id"]
    ).all()
if "product_variant_id" in reloaded_gold["fact_return_items.csv"].columns:
    reloaded_return_variant_ids = reloaded_gold["fact_return_items.csv"].loc[
        reloaded_gold["fact_return_items.csv"]["product_variant_id"].notna(),
        "product_variant_id",
    ]
    assert reloaded_return_variant_ids.isin(
        reloaded_gold["dim_product.csv"]["product_variant_id"]
    ).all()

reloaded_paid_orders = reloaded_gold["fact_orders.csv"][
    reloaded_gold["fact_orders.csv"]["status"].eq("paid")
]
assert len(reloaded_paid_orders) == 34_365
assert round(reloaded_paid_orders["total"].sum(), 2) == 985_741_294.26
assert round(reloaded_paid_orders["discount_amount"].sum(), 2) == 21_692_622.12
assert reloaded_gold["dim_employee.csv"]["hire_date"].str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all(), "dim_employee.csv: hire_date fora do padrão ISO."
assert reloaded_gold["dim_employee.csv"]["termination_date"].dropna().str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all(), "dim_employee.csv: termination_date fora do padrão ISO."
assert len({path.name for path in GOLD_DATA_DIR.glob("*.csv")}) == 12

gold_validation_report = pd.DataFrame(gold_validation_records)
print("Validação pós-exportação concluída sem erros.")

Validação pós-exportação concluída sem erros.


# 14. Resumo final

Resumo dos arquivos Gold exportados e validados.

In [58]:
print(gold_validation_report.to_string(index=False))

                                                                                                                                path  size_bytes   rows  columns                key status
            /Users/leonardoramos/Documents/dev/Desafios/indicium/desafio-lighthouse-lh-nautical/data/2-lh_nautical_gold/dim_date.csv      237597   2922       16           date_key  valid
        /Users/leonardoramos/Documents/dev/Desafios/indicium/desafio-lighthouse-lh-nautical/data/2-lh_nautical_gold/dim_customer.csv      194912   2000        9        customer_id  valid
         /Users/leonardoramos/Documents/dev/Desafios/indicium/desafio-lighthouse-lh-nautical/data/2-lh_nautical_gold/dim_product.csv      200486   1009       24 product_variant_id  valid
        /Users/leonardoramos/Documents/dev/Desafios/indicium/desafio-lighthouse-lh-nautical/data/2-lh_nautical_gold/dim_location.csv         757      6       12        location_id  valid
        /Users/leonardoramos/Documents/dev/Desafios/indicium/desa